# 🔬 ViT Fine-tuning on CIFAR-10
**COMPSCI 714 - Pretrained Models in CV**  
**Objective**: Fine-tune a Vision Transformer (ViT) on the CIFAR-10 dataset using PyTorch and Hugging Face Transformers, then evaluate and reflect using core concepts discussed in lectures.

## Tutorial Overview and Background

In this tutorial, we fine-tune a pretrained **Vision Transformer (ViT)** for image classification on **CIFAR-10**. The goal is not only to run the code, but also to understand the main ideas behind modern pretrained vision models.

By the end of this notebook, students should be able to explain:

1. what fine-tuning means in computer vision;
2. why pretrained models are useful when the target dataset is relatively small;
3. how a Vision Transformer represents an image using patches;
4. how CIFAR-10 is prepared for a pretrained ViT model;
5. how training loss, test accuracy, data augmentation, and freezing the backbone affect model behavior.


### Background 1: What is fine-tuning?

**Fine-tuning** means starting from a model that has already learned useful general features from a large dataset, and then continuing training it on a smaller target dataset for a specific task.

In this notebook, the pretrained ViT already has general visual knowledge from pretraining. We adapt it to classify CIFAR-10 images into 10 categories such as airplane, automobile, bird, cat, deer, dog, frog, horse, ship, and truck.

There are two common transfer learning strategies:

- **Full fine-tuning**: update most or all model parameters.
- **Feature extraction / frozen backbone**: freeze the pretrained backbone and train only the final classification head.

Full fine-tuning can adapt the model more strongly, but it is slower and may overfit on small datasets. Freezing the backbone is faster and often useful when the pretrained representation is already strong.


### Background 2: What is a Vision Transformer?

A **Vision Transformer**, or **ViT**, applies the Transformer idea to images. Instead of processing an image with convolutional filters, ViT divides the image into small fixed-size patches.

For the model used here, `vit-base-patch16-224`, the expected input image size is **224 × 224**, and the patch size is **16 × 16**. So the image is split into a sequence of patches. These patch embeddings are processed by Transformer self-attention layers.

The key intuition is:

- a CNN focuses heavily on local patterns through convolution;
- a ViT treats image patches like a sequence of visual tokens;
- self-attention allows the model to compare different patches and learn long-range relationships.

This is why patch size, input resolution, and positional encoding matter for ViT models.


### Background 3: What is CIFAR-10?

**CIFAR-10** is a classic image classification dataset. It contains 10 object categories and is commonly used for teaching and benchmarking computer vision models.

The original CIFAR-10 images are very small: **32 × 32** pixels. However, the pretrained ViT model used in this tutorial expects **224 × 224** images. Therefore, we resize CIFAR-10 images before feeding them into the model.

This is an important practical point: when using pretrained models, the input preprocessing must match what the model expects as closely as possible.


### Code Block: Environment setup

This first code block installs the required Python libraries in Colab. These libraries provide the pretrained ViT model, dataset utilities, image transformations, plotting tools, and progress bars.

In Colab, installed packages may disappear after the runtime is restarted, so it is common to keep installation commands at the top of the notebook.


In [ ]:
!pip install -q transformers datasets torchvision matplotlib tqdm

### Code Block: Imports and reproducibility

This code block imports the main libraries used in the tutorial:

- `torch` and `torchvision` for deep learning and image processing;
- `transformers` for loading the pretrained ViT model;
- `DataLoader` for batching data;
- `matplotlib` for visualization;
- `tqdm` for progress bars.

It also fixes the random seed. This makes the tutorial more reproducible, so students are less likely to see completely different results every time they run the notebook.


In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from transformers import ViTForImageClassification
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import random
import os

# Set random seed for reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


### Why do we fix the random seed?

Deep learning experiments contain randomness: parameter initialization, data shuffling, and GPU operations can all affect the result. Setting a fixed seed helps make the experiment easier to reproduce and debug.

This does not guarantee perfectly identical results on every machine, but it makes the behavior much more stable for a tutorial setting.


## 📊 Load and Visualize CIFAR-10

Before training, we first load the dataset and visualize a few examples. This helps students connect the labels with real images and confirm that the dataset pipeline works correctly.

This notebook first tries to load CIFAR-10 from a local `torchvision` copy. If that is not available, it falls back to the Hugging Face CIFAR-10 dataset. This makes the notebook more robust in Colab.


### Code Block: Dataset wrapper and visualization

This code block does four things:

1. defines a small dataset wrapper so that Hugging Face CIFAR-10 can behave like a PyTorch dataset;
2. defines a helper function to load CIFAR-10;
3. creates a visualization transform that resizes images and converts them to tensors;
4. displays a small batch of images with their class names.

At this stage, we are not training yet. We are just checking that the input images and labels look reasonable.


In [ ]:
from datasets import load_dataset
from torch.utils.data import Dataset

DATA_ROOT = './data'
HF_CACHE_DIR = os.path.join(DATA_ROOT, 'hf_cache')

class HFCIFAR10(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.hf_dataset = hf_dataset
        self.transform = transform
        self.classes = self.hf_dataset.features['label'].names

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        item = self.hf_dataset[idx]
        image = item['img'] if 'img' in item else item['image']
        label = item['label']

        # Ensure the image has 3 channels.
        image = image.convert('RGB')

        if self.transform is not None:
            image = self.transform(image)

        return image, label


def get_cifar10_dataset(split='train', transform=None, root=DATA_ROOT):
    # Load CIFAR-10 with the same interface as torchvision.
    # It first tries a local torchvision CIFAR-10 copy. If that is not available,
    # it uses the Hugging Face CIFAR-10 dataset. This avoids the torchvision
    # download error caused by the official CIFAR-10 server returning HTTP 503.
    train = split == 'train'

    try:
        dataset = torchvision.datasets.CIFAR10(
            root=root,
            train=train,
            transform=transform,
            download=False
        )
        classes = dataset.classes
        print(f"Loaded CIFAR-10 from local torchvision files: {root}")
        return dataset, classes
    except Exception as torchvision_error:
        print("Local torchvision CIFAR-10 was not found. Loading CIFAR-10 from Hugging Face instead.")
        print(f"torchvision message: {torchvision_error}")

    hf_dataset = load_dataset(
        'uoft-cs/cifar10',
        split=split,
        cache_dir=HF_CACHE_DIR
    )
    dataset = HFCIFAR10(hf_dataset, transform=transform)
    classes = dataset.classes
    return dataset, classes


transform_viz = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

train_viz, classes = get_cifar10_dataset(split='train', transform=transform_viz)

def imshow(img):
    # Works for both [0, 1] images and normalized [-1, 1] images.
    img = img.detach().cpu()
    if img.min() < 0:
        img = img * 0.5 + 0.5
    img = torch.clamp(img, 0, 1)
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')
    plt.show()

dataiter = iter(DataLoader(train_viz, batch_size=4, shuffle=True))
images, labels = next(dataiter)
imshow(torchvision.utils.make_grid(images))
print(' '.join(f'{classes[labels[j].item()]:5s}' for j in range(4)))


## 🧹 Prepare Data for ViT

Now we prepare CIFAR-10 images in the format expected by the pretrained ViT model.

The key preprocessing steps are:

- resize the image to **224 × 224**;
- convert the image to a PyTorch tensor;
- normalize the pixel values.

The notebook also uses only a subset of the training set to make the tutorial faster in Colab. This is useful for teaching, but students should understand that using less data may reduce final accuracy and increase overfitting risk.


### Code Block: Transforms, subsets, and DataLoaders

This code block defines the image preprocessing pipeline and creates the training and test dataloaders.

The `DataLoader` is important because it controls batching and shuffling. During training, shuffling is useful because it prevents the model from seeing the data in the same fixed order every epoch. During testing, shuffling is unnecessary because we only evaluate performance.


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
])

trainset, classes = get_cifar10_dataset(split='train', transform=transform)
small_subset = torch.utils.data.Subset(trainset, range(len(trainset) // 10))
trainset=small_subset
testset, _ = get_cifar10_dataset(split='test', transform=transform)

trainloader = DataLoader(trainset, batch_size=32, shuffle=True)
testloader = DataLoader(testset, batch_size=32, shuffle=False)


## 🤖 Load the Pretrained ViT Model

Here we load a pretrained ViT model and adapt it to CIFAR-10.

The original pretrained model was not trained specifically for the 10 CIFAR-10 classes, so we replace the final classification layer with a new 10-class head. The backbone still contains pretrained visual knowledge, while the final head learns the mapping to CIFAR-10 labels.


### Code Block: Model, device, loss, and optimizer

This code block loads `google/vit-base-patch16-224-in21k` using Hugging Face Transformers.

Important details:

- `num_labels=10` tells the model that CIFAR-10 has 10 classes;
- `ignore_mismatched_sizes=True` allows the classification head to be replaced;
- `cuda` is used if a GPU is available;
- `CrossEntropyLoss` is used for multi-class classification;
- `AdamW` is a common optimizer for Transformer-based models.

Printing the parameter names is useful for understanding the model structure, especially when discussing which parts belong to the backbone and which part is the classification head.


In [ ]:
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=10,
    ignore_mismatched_sizes=True
)
for name, param in model.named_parameters():
    print(name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-5)


## 🏋️ Training and Evaluation Functions

Training and evaluation are separated into two functions.

During training, the model updates its parameters using gradients. During evaluation, we turn off gradient computation because we only want to measure performance, not update the model.


### Code Block: Training loop and evaluation loop

The `train` function follows the standard deep learning workflow:

1. move images and labels to the GPU or CPU;
2. reset gradients;
3. run the model forward;
4. compute loss;
5. backpropagate gradients;
6. update model parameters.

The `evaluate` function uses `torch.no_grad()` to reduce memory usage and computes classification accuracy.

The argument `interpolate_pos_encoding=True` is included so that the model can still run when we later experiment with different input image sizes.


In [ ]:
def train(model, loader):
    model.train()
    total_loss = 0
    for images, labels in tqdm(loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        # interpolate_pos_encoding=True allows Q3 to run when images are resized to 96x96.
        outputs = model(images, interpolate_pos_encoding=True).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in tqdm(loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images, interpolate_pos_encoding=True).logits
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total


### Code Block: Run fine-tuning

This code block runs the training function for one epoch and prints the average training loss.

For a tutorial, one epoch is often enough to demonstrate the workflow quickly. In a real experiment, we would usually train for more epochs and monitor both training and validation performance.


In [ ]:
for epoch in range(1):
    loss = train(model, trainloader)
    # acc = evaluate(model, testloader)
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}")

## 📈 Final Evaluation and Error Analysis

After training, we evaluate the model on the test set.

Accuracy gives a simple overall measure of performance, but error analysis is also important. By looking at misclassified images, students can understand what kinds of visual examples the model finds difficult.


### Code Block: Test accuracy and misclassified examples

This code block runs the model on the test set, computes accuracy, and stores misclassified examples.

Displaying several mistakes is useful because model evaluation should not only be numerical. Visual inspection can reveal patterns such as similar classes, ambiguous images, low-resolution objects, or preprocessing issues.


In [ ]:
# Collect predictions for error analysis
from tqdm import tqdm

model.eval()
misclassified = []
correct = 0
total = 0

with torch.no_grad():
    for images, labels in tqdm(testloader):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images, interpolate_pos_encoding=True).logits
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        for img, pred, label in zip(images, preds, labels):
            if pred != label:
                misclassified.append((img.cpu(), pred.cpu(), label.cpu()))

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")


# Display up to 4 misclassified images
for i in range(min(4, len(misclassified))):
    img, pred, label = misclassified[i]
    print(f"Predicted: {classes[pred.item()]}, Actual: {classes[label.item()]}")
    imshow(img)


## ❓ Lecture-Based Reflection Questions

The following questions connect the tutorial to key lecture concepts: overfitting, data augmentation, patch sensitivity, and transfer learning efficiency.

These tasks are designed for students to modify the notebook and observe how model behavior changes. The goal is not only to get a higher accuracy, but also to explain why the performance changes.


### Concept: Overfitting and small datasets

When the training set becomes very small, a large pretrained model can still fit the training examples, but it may not generalize well to unseen test images. This is called **overfitting**.

A common warning sign is that training loss continues to decrease, while test accuracy stops improving or becomes unstable.


#Q1: Overfitting & Small Datasets (Lecture: Medical Image Use Case)
Reduce the size of the training set to 10% of original. Do you notice overfitting?

Visual clue: training loss ↓ while test accuracy stalls.

💬 Reflect: Why do smaller datasets increase overfitting risk (e.g., memorization of noisy features)?

In [ ]:
# Q1: Reduce training set size to 10%
small_subset = torch.utils.data.Subset(trainset, range(len(trainset) // 100))
small_loader = DataLoader(small_subset, batch_size=32, shuffle=True)
# Retrain with limited data to observe overfitting

### Concept: Data augmentation

Data augmentation creates modified versions of training images, such as flipped or cropped images. It encourages the model to learn more robust features instead of memorizing exact pixel patterns.

However, augmentation must be chosen carefully. Good augmentations preserve the class label, while bad augmentations can distort the image in a way that makes the label confusing.


#Q2: Data Augmentation (Lecture: Generalization Challenge)

Retrain using augmented data. Does test accuracy improve?

💬 Reflect: Which lecture examples (e.g., dog/cat confusion due to memorized color) does this relate to?

In [ ]:
# Q2: Add data augmentation
augmented_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
# Recreate dataset with augmentation and compare accuracy


### Concept: Patch sensitivity in ViT

ViT processes images as a sequence of patches. When we change the input resolution, we also change the number of visual tokens and the amount of detail available to the model.

Reducing image size may make training faster, but it can also remove useful visual information and make spatial relationships harder to learn.


#Q3: Patch Sensitivity (Lecture: Patch → Position → Attention)
Change patch size indirectly by resizing images to 96x96 instead of 224x224.

💬 Reflect: ViT splits into patches — why might reducing input size hurt its ability to capture spatial relationships?

In [ ]:
# Q3: Resize images to 96x96 and analyze performance
# This simulates patch-level impact in ViT

### Concept: Frozen backbone and transfer learning efficiency

Freezing the backbone means the pretrained ViT feature extractor is not updated. Only the final classification head is trained.

This is computationally cheaper and can reduce overfitting, especially when the target dataset is small. However, if the target data is very different from the pretraining data, full fine-tuning may be more effective.


#Q4: Frozen Backbone (Lecture: Transfer Learning Efficiency)
Only fine-tune the classification head.

💬 Reflect: How does this relate to lecture ideas of pretraining large models and adapting to new domains?

In [ ]:
# Q4: Freeze feature extractor
for param in model.vit.parameters():
    param.requires_grad = False
# Train only the classification head

## ✍️ Summary

In this tutorial, you fine-tuned a Vision Transformer on CIFAR-10 and connected the code to several important computer vision concepts.

You practiced:

- loading and visualizing an image dataset;
- preparing images for a pretrained ViT model;
- replacing the classification head for a new task;
- training and evaluating a Transformer-based vision model;
- interpreting accuracy and misclassified examples;
- reasoning about overfitting, data augmentation, patch size, and frozen backbones.

**Deliverables:**

1. final accuracy numbers;
2. answers to the reflection questions;
3. visual examples of predictions and mistakes;
4. a short 200–300 word reflection on what you learned.
